<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/01_hello_openimpala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 1: Getting Started with OpenImpala

OpenImpala computes effective transport properties (diffusivity, tortuosity, conductivity) directly on 3D voxel images of porous microstructures. It uses finite differences on the voxel grid — no mesh generation required — and is parallelised via MPI through AMReX and HYPRE.

This tutorial introduces the core workflow using a synthetic microstructure generated with [PoreSpy](https://porespy.org/), a widely used library for porous media analysis. No external data files are needed.

**What you will learn:**
1. Generate a random 3D porous microstructure with PoreSpy.
2. Calculate volume fraction, check percolation, and compute tortuosity.
3. Run a parametric sweep of porosity vs. tortuosity.

In [ ]:
# Install OpenImpala, PoreSpy for structure generation, and Matplotlib for plots.
!pip install -q openimpala porespy matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded and ready.")

# OpenImpala relies on AMReX and MPI under the hood.  We open a global
# session here so that the runtime stays alive across notebook cells.
global_session = oi.Session()
global_session.__enter__()

## 1. Generating a Synthetic Microstructure

PoreSpy's `blobs` generator creates random two-phase structures that resemble real porous media (e.g. electrode microstructures from X-ray tomography). The `blobiness` parameter controls the characteristic feature size — higher values produce coarser structures.

We generate a $100^3$ voxel grid (~1 million voxels) with a target porosity of 0.40:
- **Phase 0** — solid matrix
- **Phase 1** — void / pore space

In [ ]:
N = 100
np.random.seed(42)  # For reproducibility

micro = ps.generators.blobs(
    shape=[N, N, N],
    porosity=0.40,
    blobiness=1.5,
).astype(np.int32)

print(f"Microstructure shape: {micro.shape} ({micro.size / 1e6:.1f} million voxels)")
print(f"Measured porosity:    {micro.mean():.4f}")

A mid-plane cross-section of the generated structure. Black = solid, white = pore. The random morphology is qualitatively similar to what you would see in a segmented tomography image.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(micro[N // 2, :, :], cmap="gray", interpolation="nearest")
ax.set_title(f"Mid-plane slice (z = {N // 2})")
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Computing Transport Properties

We now pass the NumPy array directly into OpenImpala. The Python bindings transfer the array into an AMReX `iMultiFab`, formulate the steady-state diffusion problem, and solve it with HYPRE.

We set `max_grid_size=128` because our $100^3$ domain fits comfortably in memory on a single node. This avoids unnecessary domain decomposition overhead for small problems. For guidance on tuning this parameter at larger scales, see [Tutorial 7](07_hpc_scaling.ipynb).

In [ ]:
print("Starting physics calculations...")

# 1. Volume fraction
vf = oi.volume_fraction(micro, phase=1)
print(f"\nPorosity (phase 1 VF): {vf.fraction:.4f}")

# 2. Percolation check (does the pore space span the domain?)
t0 = time.time()
perc = oi.percolation_check(micro, phase=1, direction="z")
print(f"Percolates in Z:       {perc.percolates} ({time.time()-t0:.3f}s)")

# 3. Solve for tortuosity
if perc.percolates:
    t0 = time.time()
    result = oi.tortuosity(micro, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
    t_solve = time.time() - t0

    print(f"\nTortuosity (tau):      {result.tortuosity:.4f}")
    print(f"Solver iterations:     {result.iterations} (Residual: {result.residual_norm:.2e})")
    print(f"Wall time:             {t_solve:.2f}s for {micro.size / 1e6:.1f}M voxels")
else:
    print("Phase does not percolate -- tortuosity is undefined.")

## 3. Parametric Sweep: Tortuosity vs. Porosity

Because OpenImpala operates directly on NumPy arrays, scripting parametric studies is straightforward. Below we sweep over a range of porosities. At low porosity the pore network may become disconnected and the phase will not percolate — in that case, tortuosity is undefined and we skip that point.

In [ ]:
porosities = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
results = []  # (porosity, tau) pairs

print("Running parametric sweep...")
sweep_start = time.time()

for target in porosities:
    # Use a smaller grid (64^3) for the sweep to keep run times short
    data = ps.generators.blobs(
        shape=[64, 64, 64], porosity=target, blobiness=1.5,
    ).astype(np.int32)

    perc = oi.percolation_check(data, phase=1, direction="z")
    if perc.percolates:
        res = oi.tortuosity(data, phase=1, direction="z", solver="flexgmres", max_grid_size=128)
        results.append((target, res.tortuosity))
        print(f"  Porosity={target:.2f}  ->  Tau={res.tortuosity:.4f}")
    else:
        print(f"  Porosity={target:.2f}  ->  Does not percolate (skipped)")

print(f"\nEvaluated {len(porosities)} microstructures in {time.time() - sweep_start:.2f}s.")

# Plot the sweep results
if results:
    phi_vals, tau_vals = zip(*results)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    ax.plot(phi_vals, tau_vals, "o-", color="#d62728", linewidth=2.5, markersize=8,
            markerfacecolor="white", markeredgewidth=2)

    ax.set_xlabel(r"Porosity ($\varepsilon$)", fontsize=12)
    ax.set_ylabel(r"Tortuosity Factor ($\tau$)", fontsize=12)
    ax.set_title("Random Porous Media: Tortuosity vs. Porosity")
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

## Next Steps

In this tutorial we generated a microstructure, solved for its transport properties, and ran a parametric sweep — all from a single Python session, with no mesh generation or input files needed.

The remaining tutorials build on this foundation:
- [Tutorial 2: From 3D Image to Device Model](02_digital_twin.ipynb) — Load a real CT scan, measure anisotropy, and export parameters to PyBaMM.
- [Tutorial 3: REV and Uncertainty](03_rev_and_uncertainty.ipynb) — Determine whether your sample volume is statistically representative.
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Generate training data for machine learning models.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Use OpenImpala inside an optimisation loop.